In [1]:
# 1. Clona il repository

!git clone https://github.com/baddy2002/image-gs

%cd image-gs

Cloning into 'image-gs'...
remote: Enumerating objects: 1863, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 1863 (delta 111), reused 131 (delta 87), pack-reused 1682 (from 1)
Receiving objects: 100% (1863/1863), 8.73 MiB | 14.07 MiB/s, done.
Resolving deltas: 100% (1036/1036), done.
/content/image-gs


In [ ]:
# 3. Installazione dipendenze base da environment.yml
!pip install pytorch-msssim==1.0.0 torchmetrics==1.5.2 lpips==0.1.4 flip-evaluator plyfile

In [11]:
# 4. Installiamo il gsplat locale
%cd /content/image-gs/gsplat
#%cd ./gsplat
!pip install -e . -v
%cd ..

/content/image-gs/gsplat
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Obtaining file:///content/image-gs/gsplat
  Running command python setup.py egg_info
  running egg_info
  creating /tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info
  writing /tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/requires.txt
  writing top-level names to /tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/top_level.txt
  writing manifest file '/tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-pip-egg-info-xvmj95bp/gsplat.egg-info/SOURCES.txt'
  Preparing metadata (setup.py) ... done
  Obtaining dependency information for jaxtyping from https://files.pythonhosted.org/packages/94/

In [12]:

# 6. Test finale
!python main.py --help

usage: main.py [-h] [--seed SEED] [--device DEVICE] [--eval]
               [--render_height RENDER_HEIGHT] [--quantize]
               [--pos_bits POS_BITS] [--scale_bits SCALE_BITS]
               [--rot_bits ROT_BITS] [--feat_bits FEAT_BITS]
               [--beta_bits BETA_BITS] [--log_root LOG_ROOT]
               [--exp_name EXP_NAME] [--log_level LOG_LEVEL]
               [--save_image_format SAVE_IMAGE_FORMAT]
               [--save_plot_format SAVE_PLOT_FORMAT] [--vis_gaussians]
               [--save_image_steps SAVE_IMAGE_STEPS]
               [--save_ckpt_steps SAVE_CKPT_STEPS] [--eval_steps EVAL_STEPS]
               [--gamma GAMMA] [--data_root DATA_ROOT]
               [--input_path INPUT_PATH] [--downsample]
               [--downsample_ratio DOWNSAMPLE_RATIO]
               [--num_gaussians NUM_GAUSSIANS] [--init_scale INIT_SCALE]
               [--topk TOPK] [--disable_topk_norm] [--disable_inverse_scale]
               [--ckpt_file CKPT_FILE] [--disable_color_init]
 

In [13]:
# Installa lo strumento di conversione
!apt-get install imagemagick -y

!ls -lh media/textures/castpol01.png

# Ridimensiona il vaso a 1024px (molto più gestibile per una T4)
#!convert "media/textures/baron seutin01.png" -resize 1024x1024 "media/textures/baron seutin01_small.png"

# Verifica la nuova dimensione (dovrebbe essere pochi KB/MB invece di 32MB)
#!ls -lh media/textures/baron seutin01_small.png

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
imagemagick is already the newest version (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
-rw-r--r-- 1 root root 32M Apr 23 22:07 media/textures/castpol01.png


Lancia con quantizzazione 10000 gaussiane. (no mask)


In [ ]:
!python run_experiments.py

In [ ]:
from google.colab import userdata
import os
import subprocess
import pandas as pd
import glob
from PIL import Image
import re

try:
    import wandb
    HAS_WANDB = True
except ImportError:
    HAS_WANDB = False

# --- CONFIGURAZIONE ---
csv_path = "experiment_results.csv"
input_dir = "media/textures"
small_dir = "media/textures_small"
os.makedirs(small_dir, exist_ok=True)

images = glob.glob(f"{input_dir}/*.png")[:30]
densities = [1000, 2500, 5000, 10000, 20000]
betas = [1, 2, 4, 8, 10]

# --- 1. LOGIN E RESUME DA WANDB ---
completed = set()
results = []
run = None

if HAS_WANDB:
    try:
        wandb_key = userdata.get('WANDB_API_KEY')
        wandb.login(key=wandb_key)

        # Iniziamo il run
        run = wandb.init(project="beta-splatting-texture", name="marathon_run_final", resume=True)

        # Tentativo di recupero CSV dal cloud
        try:
            artifact = run.use_artifact('experiment_results:latest')
            artifact_dir = artifact.download()
            df_existing = pd.read_csv(os.path.join(artifact_dir, csv_path))

            # Popoliamo i completati per saltarli
            completed = set(zip(df_existing['texture'], df_existing['gaussians'], df_existing['beta']))
            results = df_existing.to_dict('records')
            df_existing.to_csv(csv_path, index=False) # Copia locale di sicurezza
            print(f"Cloud Resume: {len(completed)} esperimenti saltati.")
        except:
            print("Nessun artifact trovato. Parto da zero o da file locale.")
            if os.path.exists(csv_path):
                df_local = pd.read_csv(csv_path)
                completed = set(zip(df_local['texture'], df_local['gaussians'], df_local['beta']))
                results = df_local.to_dict('records')

    except Exception as e:
        print(f"⚠️ Errore WandB: {e}")
        HAS_WANDB = False

# --- 2. LOOP ESPERIMENTI ---
for img_path in images:
    img_name = os.path.basename(img_path)
    small_path = os.path.join(small_dir, img_name)

    if not os.path.exists(small_path):
        with Image.open(img_path) as img:
            img = img.resize((1024, 1024), Image.LANCZOS)
            img.save(small_path)

    for n in densities:
        for b in betas:
            if (img_name, n, b) in completed:
                continue

            exp_name = f"exp_{img_name.split('.')[0]}_n{n}_b{b}"
            print(f"\n>>> Running: {exp_name}")
            input_path_for_cmd = f"textures_small/{img_name}"

            # NB: Assicurati che main.py accetti questo percorso relativo
            cmd = ["python", "main.py",
                   f"--input_path={input_path_for_cmd}",
                   f"--exp_name={exp_name}",
                   f"--num_gaussians={str(n)}",
                   f"--beta_value={str(b)}",
                   "--quantize", "--use_mask"]

            try:
                process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

                final_metrics = {}
                for line in process.stdout:
                    print(line, end="")
                    if "PSNR:" in line and "LPIPS:" in line:
                        try:
                            psnr_match  = re.search(r'PSNR:\s*([\d.]+)', line)
                            ssim_match  = re.search(r'SSIM:\s*([\d.]+)', line)
                            lpips_match = re.search(r'LPIPS:\s*([\d.]+)', line)

                            if psnr_match and ssim_match and lpips_match:
                                final_metrics = {
                                    "texture":   img_name,
                                    "gaussians": n,
                                    "beta":      b,
                                    "psnr":      float(psnr_match.group(1)),
                                    "ssim":      float(ssim_match.group(1)),
                                    "lpips":     float(lpips_match.group(1))
                                }
                        except Exception as parse_err:
                            print(f"Parse error: {parse_err} on line: {line}")
                            continue
                process.wait()

                if final_metrics:
                    results.append(final_metrics)
                    # Aggiorna CSV locale
                    pd.DataFrame(results).to_csv(csv_path, index=False)

                    # Log su WandB (Dati finali)
                    if HAS_WANDB:
                        wandb.log(final_metrics)
                        # Ogni 10 esperimenti facciamo il backup del CSV sul cloud
                        if len(results) % 10 == 0:
                            new_art = wandb.Artifact('experiment_results', type='dataset')
                            new_art.add_file(csv_path)
                            run.log_artifact(new_art)

            except Exception as e:
                print(f"Errore critico su {exp_name}: {e}")
                continue

if HAS_WANDB:
    # Caricamento finale del CSV completo
    final_art = wandb.Artifact('experiment_results', type='dataset')
    final_art.add_file(csv_path)
    run.log_artifact(final_art)
    wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: andreabenassi02 (andreabenassi02-university-of-modena) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb:   1 of 1 files downloaded.  


Cloud Resume: 240 esperimenti saltati.

>>> Running: exp_Model_0_n1000_b1
[2026/04/23 22:12:04] Start optimizing 1000 Gaussians for 'textures_small/Model_0.png'
[2026/04/23 22:12:04] ***********************************************
Maschera creata con Flood Fill. Pixel attivi: 895831/1048576
[2026/04/23 22:12:05] Spawn Mask: Utilizzo maschera UV precisa.
[2026/04/23 22:12:05] Uncompressed: 3145.73 KB | 24.000 bpp | 8.000 bppc
[2026/04/23 22:12:05] Compressed: 18.00 KB | 0.137 bpp | 0.046 bppc
[2026/04/23 22:12:05] Compression rate: 174.76x | 0.57%
[2026/04/23 22:12:05] ***********************************************
[2026/04/23 22:12:05] Image gradient map successfully saved
[2026/04/23 22:12:05] ***********************************************
[2026/04/23 22:12:10] Step: 100 | Total: 2.89 s | Render: 0.23 s | Loss: 0.3950, L1: 0.2966, SSIM: 0.0984 | PSNR: 9.10 | SSIM: 0.0167
[2026/04/23 22:12:13] Step: 200 | Total: 5.90 s | Render: 0.48 s | Loss: 0.2263, L1: 0.1385, SSIM: 0.0878 | PSNR: